# AVION + SMS Loss — EPIC-KITCHENS-100 MIR

Replicates the 2024 winning approach (~74% nDCG) using:
- **AVION** backbone (CLIP ViT-L pre-trained on Ego4D via LaViLa)
- **SMS Loss** (Symmetric Multi-Similarity Loss with relevancy matrix)

## Prerequisites — do these BEFORE running any cell

### 1. Runtime: GPU → A100
Runtime → Change runtime type → A100

### 2. Add AVION pre-processed videos to your Drive
Open this link and click **"Add shortcut to Drive"** → My Drive:
```
https://drive.google.com/file/d/13J2uC2g2H_DEHrBvgr5Aiu0BgqlCvWqG/view
```
If it is a zip file, also run the extraction cell (Cell 5b). The folder must be named `EK100_320p_15sec_30fps_libx264`.

### 3. Upload annotation files to your Drive
From your local machine, upload these files to `MyDrive/EK100_annotations/`:
```
EK100_MIR/data/MI-MM/dataframes/EPIC_100_retrieval_train.csv
EK100_MIR/data/MI-MM/dataframes/EPIC_100_retrieval_test.csv
EK100_MIR/data/MI-MM/relevancy/caption_relevancy_EPIC_100_retrieval_train.pkl
```
The test relevancy (`caption_relevancy_EPIC_100_retrieval_test.pkl`) is bundled
inside the AVION GDrive archive. If missing, local nDCG metrics will be skipped
but the submission file is still generated.

In [1]:
# Cell 1 — GPU check
!nvidia-smi
import torch
print(f"PyTorch {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

Tue May  5 18:44:52 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   31C    P0             53W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [2]:
# Cell 2 — Install dependencies
!apt-get install -qq ffmpeg libavcodec-dev libavformat-dev libavutil-dev libswscale-dev

# Remove flash_attn if it is already installed — any pre-installed version is
# likely compiled for a different torch version and will crash with undefined symbols.
!pip uninstall -q -y flash-attn flash_attn 2>/dev/null; echo "flash_attn removed (or was not present)"

!pip install -q einops kornia timm transformers

# decord: try standard, fall back to eva-decord (maintained fork, same namespace)
import subprocess, sys
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "decord"],
                   capture_output=True, text=True)
if r.returncode != 0:
    print("decord failed, falling back to eva-decord...")
    !pip install -q eva-decord
else:
    print("decord installed.")

!pip install -q git+https://github.com/openai/CLIP.git

import importlib
HAS_FLASH_ATTN = importlib.util.find_spec("flash_attn") is not None
print(f"flash_attn available: {HAS_FLASH_ATTN}")   # should be False
print("Done.")

flash_attn removed (or was not present)
decord installed.
  Preparing metadata (setup.py) ... done
flash_attn available: False
Done.


In [3]:
!pip install ninja

In [4]:
!pip install torch=='2.4.1+cu121' torchvision=='0.19.1+cu121' torchaudio=='2.4.1+cu121' --index-url https://download.pytorch.org/whl/cu121


Looking in indexes: https://download.pytorch.org/whl/cu121


In [5]:
!pip install flash-attn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 73.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for flash-attn: filename=flash_attn-2.8.3-cp312-cp312-linux_x86_64.whl size=255985226 sha256=ee1fbb7dc9d4f6e973687e15e245727d39c2b5f5884c74fa30d21bd1841854af
  Stored in directory: /root/.cache/pip/wheels/3d/59/46/f282c12c73dd4bb3c2e3fe199f1a0d0f8cec06df0cccfeee27
Successfully built flash-attn


In [6]:
import importlib
HAS_FLASH_ATTN = importlib.util.find_spec("flash_attn") is not None
print(f"flash_attn available: {HAS_FLASH_ATTN}")   # should be False
print("Done.")

flash_attn available: True
Done.


In [7]:
# Cell 3 — Mount Google Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [8]:
# Cell 4 — Configure paths
import os

GDRIVE = "/content/drive/MyDrive"

# Path to the AVION pre-processed EK-100 videos (320p, 15-sec chunks)
# Must contain participant folders: P01/, P02/, ..., P37/
DATA_ROOT = f"{GDRIVE}/EK100_320p_15sec_30fps_libx264"

# Annotation files — upload from local EK100_MIR/data/ before running
ANNOT_DIR = f"{GDRIVE}/EK100_annotations"

# Experiment output (saved to Drive so it survives session disconnect)
EXP_DIR = f"{GDRIVE}/experiments/sms_vitl"

# AVION ViT-L pretrain checkpoint (downloaded in Cell 6)
PRETRAIN_CKPT = f"{GDRIVE}/checkpoints/avion_pretrain_lavila_vitl_best.pt"

os.makedirs(ANNOT_DIR, exist_ok=True)
os.makedirs(EXP_DIR, exist_ok=True)
os.makedirs(os.path.dirname(PRETRAIN_CKPT), exist_ok=True)

print(f"DATA_ROOT : {DATA_ROOT}")
print(f"  exists  : {os.path.isdir(DATA_ROOT)}")
print(f"ANNOT_DIR : {ANNOT_DIR}")
print(f"EXP_DIR   : {EXP_DIR}")

DATA_ROOT : /content/drive/MyDrive/EK100_320p_15sec_30fps_libx264
  exists  : True
ANNOT_DIR : /content/drive/MyDrive/EK100_annotations
EXP_DIR   : /content/drive/MyDrive/experiments/sms_vitl


In [9]:
# Cell 5a — Check video data
# The AVION GDrive link might be a folder shortcut OR a zip file.
# After adding the shortcut, check whether it's already a directory:
if os.path.isdir(DATA_ROOT):
    participants = [d for d in os.listdir(DATA_ROOT) if d.startswith('P')]
    print(f"Found {len(participants)} participant folders: {sorted(participants)[:5]}...")
else:
    print("DATA_ROOT not found as a directory.")
    # Check if the shortcut landed as a zip:
    zip_path = f"{GDRIVE}/EK100_320p_15sec_30fps_libx264.zip"
    if os.path.isfile(zip_path):
        print(f"Found zip at {zip_path}. Run Cell 5b to extract.")
    else:
        print("Neither folder nor zip found. Please add the GDrive shortcut first.")
        print("Link: https://drive.google.com/file/d/13J2uC2g2H_DEHrBvgr5Aiu0BgqlCvWqG/view")

Found 37 participant folders: ['P01', 'P02', 'P03', 'P04', 'P05']...


In [10]:
# Cell 6 — Download AVION LaViLa ViT-L pretrain checkpoint (~1.3 GB)
# Source: https://github.com/zhaoyue-zephyrus/AVION/blob/main/scripts/download_checkpoints.sh
if not os.path.isfile(PRETRAIN_CKPT):
    print("Downloading AVION ViT-L pretrain checkpoint...")
    !wget --show-progress -O "$PRETRAIN_CKPT" \
        "https://utexas.box.com/shared/static/1iatmrs7ufdeooce09a61t1n6wsouf4l.pt"
else:
    print(f"Checkpoint already present ({os.path.getsize(PRETRAIN_CKPT)/1e9:.2f} GB).")

Checkpoint already present (5.12 GB).


In [11]:
# Cell 7 — Verify annotation files
# These should have been uploaded from local EK100_MIR/data/ to ANNOT_DIR on Drive.
# If any are missing, this cell downloads them from EPIC-KITCHENS GitHub.
import subprocess

TRAIN_CSV = f"{ANNOT_DIR}/EPIC_100_retrieval_train.csv"
TEST_CSV  = f"{ANNOT_DIR}/EPIC_100_retrieval_test.csv"
TRAIN_REL = f"{ANNOT_DIR}/caption_relevancy_EPIC_100_retrieval_train.pkl"
# Test relevancy is bundled inside the AVION GDrive archive under:
#   epic-kitchens-100-annotations/retrieval_annotations/relevancy/
# Try that path first, then fall back to ANNOT_DIR.
TEST_REL_AVION = (
    f"{DATA_ROOT}/epic-kitchens-100-annotations/"
    "retrieval_annotations/relevancy/"
    "caption_relevancy_EPIC_100_retrieval_test.pkl"
)
TEST_REL = TEST_REL_AVION if os.path.isfile(TEST_REL_AVION) else f"{ANNOT_DIR}/caption_relevancy_EPIC_100_retrieval_test.pkl"

BASE = "https://raw.githubusercontent.com/epic-kitchens/epic-kitchens-100-annotations/master/retrieval_annotations"

for path, url in [
    (TRAIN_CSV, f"{BASE}/EPIC_100_retrieval_train.csv"),
    (TEST_CSV,  f"{BASE}/EPIC_100_retrieval_test.csv"),
]:
    if not os.path.isfile(path):
        print(f"Downloading {os.path.basename(path)} ...")
        subprocess.run(["wget", "-q", "-O", path, url], check=True)

# Train relevancy — GitHub raw works for pkl
if not os.path.isfile(TRAIN_REL):
    url = f"{BASE}/relevancy/caption_relevancy_EPIC_100_retrieval_train.pkl"
    print("Downloading train relevancy ...")
    subprocess.run(["wget", "-q", "-O", TRAIN_REL, url], check=True)

print(f"TRAIN_CSV  : {os.path.isfile(TRAIN_CSV)}")
print(f"TEST_CSV   : {os.path.isfile(TEST_CSV)}")
print(f"TRAIN_REL  : {os.path.isfile(TRAIN_REL)}")
print(f"TEST_REL   : {os.path.isfile(TEST_REL)}  (path: {TEST_REL})")

TRAIN_CSV  : True
TEST_CSV   : True
TRAIN_REL  : True
TEST_REL   : True  (path: /content/drive/MyDrive/EK100_annotations/caption_relevancy_EPIC_100_retrieval_test.pkl)


In [12]:
# Cell 8 — Clone SMS-Loss repo
import os
os.chdir('/content')
if not os.path.isdir('/content/SMS-Loss'):
    !git clone https://github.com/xqwang14/SMS-Loss.git
os.chdir('/content/SMS-Loss')
print("Working directory:", os.getcwd())
!ls scripts/

Working directory: /content/SMS-Loss
ammplus_finetune.py  dirtrain.py  ensemble.py  test_mir.py


In [13]:
HAS_FLASH_ATTN

True

In [14]:
pip install open_clip_torch


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 46.4 MB/s eta 0:00:00


In [15]:
import subprocess, os
ANNOT_DIR = "/content/drive/MyDrive/EK100_annotations"
BASE = "https://raw.githubusercontent.com/epic-kitchens/epic-kitchens-100-annotations/master/retrieval_annotations"
for fname in ["EPIC_100_retrieval_train_sentence.csv", "EPIC_100_retrieval_test_sentence.csv"]:
    path = f"{ANNOT_DIR}/{fname}"
    if not os.path.isfile(path):
        subprocess.run(["wget", "-q", "-O", path, f"{BASE}/{fname}"], check=True)
        print(f"Downloaded {fname}")
    else:
        print(f"Already present: {fname}")

Already present: EPIC_100_retrieval_train_sentence.csv
Already present: EPIC_100_retrieval_test_sentence.csv


In [16]:
# Patch ammplus_finetune.py — add images/texts unpacking in the gradient accumulation branch
path = '/content/SMS-Loss/scripts/ammplus_finetune.py'
with open(path, 'r') as f:
    src = f.read()

old = (
    '        else:\n'
    '            # First, cache the features without any gradient tracking.\n'
    '            with torch.no_grad():\n'
)
new = (
    '        else:\n'
    '            images, texts = inputs[0], inputs[1]\n'
    '            # First, cache the features without any gradient tracking.\n'
    '            with torch.no_grad():\n'
)

if old in src:
    src = src.replace(old, new)
    with open(path, 'w') as f:
        f.write(src)
    print("Patched: images/texts unpacking added to gradient accumulation branch.")
else:
    print("Pattern not found — check indentation or already patched:")
    for i, line in enumerate(src.splitlines(), 1):
        if 'First, cache the features' in line:
            print(f"  line {i}: {repr(line)}")

Patched: images/texts unpacking added to gradient accumulation branch.


In [17]:
path = '/content/SMS-Loss/scripts/ammplus_finetune.py'
with open(path, 'r') as f:
    src = f.read()

# args.accum_freq is referenced but never defined in argparse — it's the same as update_freq
count = src.count('args.accum_freq')
src = src.replace('args.accum_freq', 'args.update_freq')

with open(path, 'w') as f:
    f.write(src)
print(f"Replaced {count} occurrence(s) of args.accum_freq → args.update_freq")

Replaced 1 occurrence(s) of args.accum_freq → args.update_freq


In [18]:
path = '/content/SMS-Loss/avion/data/clip_dataset.py'
with open(path, 'r') as f:
    src = f.read()

# Patch __getitem__ to retry with a random sample if get_raw_item() returns None
old = '''    def __getitem__(self, i):
        frames, caption, pos = self.get_raw_item('''

new = '''    def __getitem__(self, i):
        import random as _random
        result = None
        _tries = 0
        while result is None and _tries < 20:
            result = self.get_raw_item('''

if old in src:
    src = src.replace(old, new)
    # Now find the closing of the get_raw_item call and add retry logic
    old2 = '''        )

        if isinstance(caption, tuple):'''
    new2 = '''        )
            _tries += 1
            if result is None:
                i = _random.randint(0, len(self) - 1)
        if result is None:
            raise RuntimeError(f"Could not load a valid sample after 20 retries")
        frames, caption, pos = result

        if isinstance(caption, tuple):'''
    src = src.replace(old2, new2, 1)  # only replace first occurrence
    with open(path, 'w') as f:
        f.write(src)
    print("Patched clip_dataset.py — None returns from get_raw_item now retry with a random index.")
else:
    print("Pattern not found. Check indentation:")
    for i, l in enumerate(src.splitlines(), 1):
        if '__getitem__' in l:
            print(f"  line {i}: {repr(l)}")

Patched clip_dataset.py — None returns from get_raw_item now retry with a random index.


In [19]:
# ── Fix __getitem__ scope bug in clip_dataset.py ─────────────────────────────
import re

path = '/content/SMS-Loss/avion/data/clip_dataset.py'
with open(path) as f:
    src = f.read()

# 1. Locate __getitem__ method boundaries
gi_s = src.index('    def __getitem__(self, i):\n')
nxt = re.search(r'\n    def [a-z_]', src[gi_s + 30:])
gi_e = gi_s + 30 + nxt.start() + 1
old = src[gi_s:gi_e]

# 2. Extract self.get_raw_item(...) using paren-depth counter
call_s = old.index('self.get_raw_item(')
pos = call_s + len('self.get_raw_item(')
depth = 1
while depth:
    c = old[pos]; pos += 1
    if c == '(':   depth += 1
    elif c == ')': depth -= 1
raw_call = old[call_s:pos]          # "self.get_raw_item(i, ..., )"

# 3. Re-indent continuation lines to 16 spaces (inside while loop)
lines = raw_call.split('\n')
reindented = lines[0]               # "self.get_raw_item("
for ln in lines[1:]:
    s = ln.strip()
    reindented += '\n' + ('                ' + s if s else '')

# 4. Post-unpacking code (anchor on "if isinstance(caption, tuple):")
post_s = old.index('        if isinstance(caption, tuple):')
post   = old[post_s:]

# 5. Build corrected method — frames,caption,pos = result is AFTER the loop
fixed = (
    '    def __getitem__(self, i):\n'
    '        import random as _random\n'
    '        result = None\n'
    '        _tries = 0\n'
    '        while result is None and _tries < 20:\n'
    '            result = ' + reindented + '\n'
    '            _tries += 1\n'
    '            if result is None:\n'
    '                i = _random.randint(0, len(self) - 1)\n'
    '        if result is None:\n'
    '            raise RuntimeError("No valid sample after 20 retries")\n'
    '        frames, caption, pos = result\n'
    + post
)

# 6. Write back and preview
with open(path, 'w') as f:
    f.write(src[:gi_s] + fixed + src[gi_e:])

with open(path) as f:
    new = f.read()
gi2 = new.index('    def __getitem__(self, i):\n')
nxt2 = re.search(r'\n    def [a-z_]', new[gi2 + 30:])
end = (gi2 + 30 + nxt2.start() + 1) if nxt2 else (gi2 + 1500)
print(new[gi2:end][:900])
print('\n✓ patch applied')

    def __getitem__(self, i):
        import random as _random
        result = None
        _tries = 0
        while result is None and _tries < 20:
            result = self.get_raw_item(
                i, is_training=self.is_training,
                chunk_len=self.chunk_len,
                clip_length=self.clip_length,
                clip_stride=self.clip_stride,
                threads=self.threads,
                fast_rrc=self.fast_rrc,
                rrc_params=self.rrc_params,
                fast_rcc=self.fast_rcc,
                rcc_params=self.rcc_params,
                )
            _tries += 1
            if result is None:
                i = _random.randint(0, len(self) - 1)
        if result is None:
            raise RuntimeError("No valid sample after 20 retries")
        frames, caption, pos = result
        if isinstance(caption, tuple):
            caption, re

✓ patch applied


In [20]:
# ── Patch ammplus_finetune.py: checkpoint every epoch + no validation ─────────
import re

path = '/content/SMS-Loss/scripts/ammplus_finetune.py'
with open(path) as f:
    src = f.read()
lines = src.split('\n')

# ── Diagnose: show key lines so we can verify patches hit the right spots ─────
print("=== Key lines ===")
for i, line in enumerate(lines):
    if any(k in line for k in ['validate_mir', 'eval_freq', 'save_on_master',
                                'save_checkpoint', 'checkpoint_path', 'output_dir']):
        print(f"{i+1:4d}│ {line}")

=== Key lines ===
 279│         latest = os.path.join(args.output_dir, 'checkpoint.pt')
 374│         val_stats = validate_mir(val_loader, val_transform_gpu, model, criterion, args)
 376│             with open(os.path.join(args.output_dir, 'eval_log.txt'), 'a') as f:
 394│         if (epoch + 1) % args.eval_freq != 0:
 397│         val_stats = validate_mir(val_loader, val_transform_gpu, model, criterion, args)
 407│         dist_utils.save_on_master({
 414│             }, is_best, args.output_dir)
 421│             with open(os.path.join(args.output_dir, 'log.txt'), 'a') as f:
 554│ def validate_mir(val_loader, transform_gpu, model, criterion, args):
 648│     os.makedirs(args.output_dir, exist_ok=True)


In [21]:
# ── Patch ammplus_finetune.py: save checkpoint every epoch, no validation ─────
path = '/content/SMS-Loss/scripts/ammplus_finetune.py'
with open(path) as f:
    lines = f.readlines()

# ── Show context so we can confirm the patch targets ─────────────────────────
print("=== Lines 368–420 ===")
for i, line in enumerate(lines[367:420], start=368):
    print(f"{i:4d}│ {line}", end='')

# ── Helpers ───────────────────────────────────────────────────────────────────
def find_line(lines, pattern, start=0):
    for i in range(start, len(lines)):
        if pattern in lines[i]:
            return i
    return -1

def indent_of(line):
    return len(line) - len(line.lstrip())

# ── Find key line indices (0-based) ──────────────────────────────────────────
mir1  = find_line(lines, 'val_stats = validate_mir')
cond  = find_line(lines, 'eval_freq != 0')
mir2  = find_line(lines, 'val_stats = validate_mir', mir1 + 1)
save  = find_line(lines, 'save_on_master')
assert all(x >= 0 for x in [mir1, cond, mir2, save]), \
    f"Pattern not found: mir1={mir1} cond={cond} mir2={mir2} save={save}"
print(f"\nmir1={mir1+1}  cond={cond+1}  mir2={mir2+1}  save={save+1}")

# ── Patch 1: Neutralise BOTH validate_mir calls ───────────────────────────────
for idx in [mir1, mir2]:
    ind = ' ' * indent_of(lines[idx])
    lines[idx] = (
        f"{ind}val_stats = {{'nDCG': 0.0, 'mAP': 0.0, 'avg_nDCG': 0.0, 'avg_mAP': 0.0}}"
        f"  # validation disabled\n"
        f"{ind}# ORIG: {lines[idx].lstrip()}"
    )

# ── Patch 2: Remove the eval_freq skip-block (replaces "if != 0: continue") ──
# Walk the if-body (lines more indented than the condition)
cond_ind = indent_of(lines[cond])
body_end = cond + 1
while body_end < len(lines):
    ln = lines[body_end]
    if ln.strip() and indent_of(ln) <= cond_ind:
        break
    body_end += 1
print(f"\neval_freq block body (lines {cond+2}–{body_end}):")
print(''.join(lines[cond+1:body_end]))
# Delete the entire if-block (condition + body) — checkpoint save below handles it
del lines[cond:body_end]

# ── Patch 3: Add numbered checkpoint save after save_on_master ───────────────
# Re-locate save_on_master after deletion
save = find_line(lines, 'save_on_master')
# Find end of the multi-line call
depth, save_end = 0, save
for i in range(save, min(save + 15, len(lines))):
    for c in lines[i]:
        if c == '(': depth += 1
        elif c == ')': depth -= 1
    if depth <= 0 and i >= save:
        save_end = i
        break
ind = ' ' * indent_of(lines[save])
numbered = [
    f"{ind}# ── per-epoch numbered checkpoint ──────────────────────────────\n",
    f"{ind}if dist_utils.is_main_process():\n",
    f"{ind}    import torch as _t\n",
    f"{ind}    _sd = model.module.state_dict() if hasattr(model, 'module') else model.state_dict()\n",
    f"{ind}    _t.save({{'epoch': epoch + 1, 'state_dict': _sd,\n",
    f"{ind}             'optimizer': optimizer.state_dict(),\n",
    f"{ind}             'scaler': scaler.state_dict()}},\n",
    f"{ind}            os.path.join(args.output_dir, f'checkpoint_{{epoch:04d}}.pt'))\n",
    f"{ind}    print(f'=> saved checkpoint_{{epoch:04d}}.pt')\n",
]
lines[save_end + 1:save_end + 1] = numbered

# ── Write back ────────────────────────────────────────────────────────────────
with open(path, 'w') as f:
    f.writelines(lines)

print("\n✓ Done. Key lines now:")
with open(path) as f:
    nl = f.readlines()
for i, line in enumerate(nl):
    if any(k in line for k in ['validate_mir', 'eval_freq', 'save_on_master',
                                'checkpoint_', 'ORIG:', 'per-epoch']):
        print(f"{i+1:4d}│ {line}", end='')

=== Lines 368–420 ===
 368│         val_dataset, batch_size=args.batch_size, shuffle=False,
 369│         num_workers=args.workers, pin_memory=False, sampler=val_sampler, drop_last=False
 370│     )
 371│     print('len(val_loader) = {}'.format(len(val_loader)))
 372│ 
 373│     if args.evaluate:
 374│         val_stats = validate_mir(val_loader, val_transform_gpu, model, criterion, args)
 375│         if dist_utils.is_main_process():
 376│             with open(os.path.join(args.output_dir, 'eval_log.txt'), 'a') as f:
 377│                 f.write(json.dumps(val_stats) + '\n')
 378│         return
 379│ 
 380│     lr_schedule = cosine_scheduler(
 381│         args.lr, args.lr_end, args.epochs, len(train_loader) // args.update_freq,
 382│         warmup_epochs=args.warmup_epochs, start_warmup_value=args.lr_start
 383│     )
 384│ 
 385│     print(args)
 386│ 
 387│     print("=> beginning training")
 388│     best_acc1 = 0.
 389│     for epoch in range(args.start_epoch, args.epochs):
 

In [22]:
path = '/content/SMS-Loss/scripts/ammplus_finetune.py'
with open(path) as f:
    src = f.read()

# Replace both occurrences of the dummy val_stats with a complete set of keys
old = "val_stats = {'nDCG': 0.0, 'mAP': 0.0, 'avg_nDCG': 0.0, 'avg_mAP': 0.0}  # validation disabled"
new = ("val_stats = {'nDCG': 0.0, 'mAP': 0.0, 'avg_nDCG': 0.0, 'avg_mAP': 0.0, "
       "'avg_map': 0.0, 'vis_map': 0.0, 'txt_map': 0.0, "
       "'vis_ndcg': 0.0, 'txt_ndcg': 0.0}  # validation disabled")

count = src.count(old)
src = src.replace(old, new)
with open(path, 'w') as f:
    f.write(src)
print(f"Fixed {count} occurrence(s)")

Fixed 2 occurrence(s)


In [ ]:
# Cell 9 — Fine-tune AVION ViT-L with SMS Loss  (single A100, ~10-14 h for 50 epochs)
import os, importlib
os.chdir('/content/SMS-Loss')

GDRIVE    = "/content/drive/MyDrive"
ANNOT_DIR = f"{GDRIVE}/EK100_annotations"
DATA_ROOT = f"{GDRIVE}/EK100_320p_15sec_30fps_libx264"
EXP_DIR   = f"{GDRIVE}/experiments/sms_vitl"
PRETRAIN_CKPT = f"{GDRIVE}/checkpoints/avion_pretrain_lavila_vitl_best.pt"

TRAIN_CSV = f"{ANNOT_DIR}/EPIC_100_retrieval_train.csv"
TEST_CSV  = f"{ANNOT_DIR}/EPIC_100_retrieval_test.csv"
TRAIN_REL = f"{ANNOT_DIR}/caption_relevancy_EPIC_100_retrieval_train.pkl"
TEST_REL_AVION = (
    f"{DATA_ROOT}/epic-kitchens-100-annotations/"
    "retrieval_annotations/relevancy/"
    "caption_relevancy_EPIC_100_retrieval_test.pkl"
)
TEST_REL = TEST_REL_AVION if os.path.isfile(TEST_REL_AVION) else f"{ANNOT_DIR}/caption_relevancy_EPIC_100_retrieval_test.pkl"

HAS_FLASH_ATTN = importlib.util.find_spec("flash_attn") is not None
flash_flag = "--use-flash-attn" if HAS_FLASH_ATTN else ""
print(f"flash_attn: {'enabled' if HAS_FLASH_ATTN else 'disabled'}")

cmd = f"""\
torchrun --nproc_per_node=1 scripts/ammplus_finetune.py \\
  --root "{DATA_ROOT}" \\
  --train-metadata "{TRAIN_CSV}" \\
  --val-metadata   "{TEST_CSV}" \\
  --relevancy-train "{TRAIN_REL}" \\
  --relevancy-test  "{TEST_REL}" \\
  --pretrain-model  "{PRETRAIN_CKPT}" \\
  --model CLIP_VITL14 \\
  --batch-size 48 \\
  --update-freq 1 \\
  --epochs 10 \\
  --lr 2e-5 \\
  --loss-margin 0.6 \\
  --loss-thres  0.1 \\
  --use-fast-conv1 \\
  {flash_flag} \\
  --grad-checkpointing \\
  --output-dir "{EXP_DIR}"
"""
print(cmd)
!{cmd}

flash_attn: enabled
torchrun --nproc_per_node=1 scripts/ammplus_finetune.py \
  --root "/content/drive/MyDrive/EK100_320p_15sec_30fps_libx264" \
  --train-metadata "/content/drive/MyDrive/EK100_annotations/EPIC_100_retrieval_train.csv" \
  --val-metadata   "/content/drive/MyDrive/EK100_annotations/EPIC_100_retrieval_test.csv" \
  --relevancy-train "/content/drive/MyDrive/EK100_annotations/caption_relevancy_EPIC_100_retrieval_train.pkl" \
  --relevancy-test  "/content/drive/MyDrive/EK100_annotations/caption_relevancy_EPIC_100_retrieval_test.pkl" \
  --pretrain-model  "/content/drive/MyDrive/checkpoints/avion_pretrain_lavila_vitl_best.pt" \
  --model CLIP_VITL14 \
  --batch-size 48 \
  --update-freq 1 \
  --epochs 10 \
  --lr 2e-5 \
  --loss-margin 0.6 \
  --loss-thres  0.1 \
  --use-fast-conv1 \
  --use-flash-attn \
  --grad-checkpointing \
  --output-dir "/content/drive/MyDrive/experiments/sms_vitl"

/content/SMS-Loss/scripts/ammplus_finetune.py:22: DeprecationWarning: `TorchScript` su

In [23]:



# ── Patch test_mir.py ────────────────────────────────────────────────────────
path = '/content/SMS-Loss/scripts/test_mir.py'
with open(path) as f:
    src = f.read()

# 1. Fix torch.load + handle missing ckpt['args']
old = (
    "    ckpt = torch.load(ckpt_path, map_location='cpu')\n"
    "    state_dict = OrderedDict()\n"
    "    for k, v in ckpt['state_dict'].items():\n"
    "        state_dict[k.replace('module.', '')] = v\n\n"
    "    old_args = ckpt['args']"
)
new = (
    "    # ── serialization compat fix ────────────────────────────────────\n"
    "    import sys, types as _t\n"
    "    if 'torch.utils.serialization' not in sys.modules:\n"
    "        _m = _t.ModuleType('torch.utils.serialization')\n"
    "        class _ST:\n"
    "            _map = {'FloatStorage': torch.FloatStorage,\n"
    "                    'DoubleStorage': torch.DoubleStorage,\n"
    "                    'HalfStorage': torch.HalfStorage,\n"
    "                    'ByteStorage': torch.ByteStorage,\n"
    "                    'CharStorage': torch.CharStorage,\n"
    "                    'ShortStorage': torch.ShortStorage,\n"
    "                    'IntStorage': torch.IntStorage,\n"
    "                    'LongStorage': torch.LongStorage,\n"
    "                    'BFloat16Storage': torch.BFloat16Storage}\n"
    "            def __new__(cls, name): return cls._map.get(name, torch.FloatStorage)\n"
    "        _m.StorageType = _ST\n"
    "        sys.modules['torch.utils.serialization'] = _m\n"
    "        import torch.utils as _tu; setattr(_tu, 'serialization', _m)\n"
    "    # ────────────────────────────────────────────────────────────────\n"
    "    ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=False)\n"
    "    state_dict = OrderedDict()\n"
    "    _sd = ckpt['state_dict'] if 'state_dict' in ckpt else ckpt\n"
    "    for k, v in _sd.items():\n"
    "        state_dict[k.replace('module.', '')] = v\n\n"
    "    old_args = ckpt.get('args', args)\n"
    "    if not hasattr(old_args, 'norm_style'): old_args.norm_style = 'openai'\n"
    "    if not hasattr(old_args, 'model'):      old_args.model = args.model"
)
src = src.replace(old, new)
assert old not in src or new in src, "Patch 1 failed to apply"

# 2. Add --project-embed-dim to argparser
old2 = "    parser.add_argument('--pretrain-model', default='', type=str, help='path of pretrained model')"
new2 = ("    parser.add_argument('--project-embed-dim', default=256, type=int)\n" + old2)
src = src.replace(old2, new2)

# 3. Make relevancy optional — save submission FIRST, then optionally compute metrics
old3 = (
    "    rel_matrix = pd.read_pickle(args.relevancy_path)\n"
    "    vis_map, txt_map, avg_map = get_mAP(similarity_matrix, rel_matrix)\n"
    "    print('mAP: V->T: {:.3f} T->V: {:.3f} AVG: {:.3f}'.format(vis_map, txt_map, avg_map))\n"
    "    vis_nDCG, txt_nDCG, avg_nDCG = get_nDCG(similarity_matrix, rel_matrix)\n"
    "    print('nDCG: V->T: {:.3f} T->V: {:.3f} AVG: {:.3f}'.format(vis_nDCG, txt_nDCG, avg_nDCG))\n\n"
    "    create_and_save_dict(similarity_matrix, text_id, video_id)"
)
new3 = (
    "    import os as _os\n"
    "    _sub = _os.path.join(args.output_dir, 'submission.pkl')\n"
    "    create_and_save_dict(similarity_matrix, text_id, video_id, filename=_sub)\n"
    "    vis_map = txt_map = avg_map = vis_nDCG = txt_nDCG = avg_nDCG = 0.0\n"
    "    _rel = getattr(args, 'relevancy_path', '')\n"
    "    if _rel and _os.path.isfile(_rel):\n"
    "        rel_matrix = pd.read_pickle(_rel)\n"
    "        vis_map, txt_map, avg_map = get_mAP(similarity_matrix, rel_matrix)\n"
    "        print('mAP: V->T: {:.3f} T->V: {:.3f} AVG: {:.3f}'.format(vis_map, txt_map, avg_map))\n"
    "        vis_nDCG, txt_nDCG, avg_nDCG = get_nDCG(similarity_matrix, rel_matrix)\n"
    "        print('nDCG: V->T: {:.3f} T->V: {:.3f} AVG: {:.3f}'.format(vis_nDCG, txt_nDCG, avg_nDCG))\n"
    "    else:\n"
    "        print(f'Submission saved to {_sub} (no relevancy file — skipping metrics)')"
)
src = src.replace(old3, new3)

with open(path, 'w') as f:
    f.write(src)
print("✓ test_mir.py patched")

✓ test_mir.py patched


In [24]:
!pip install reranking

In [25]:
path = '/content/SMS-Loss/scripts/test_mir.py'
with open(path) as f:
    src = f.read()

src = src.replace(
    'from reranking import re_ranking, naive_rerank',
    '# from reranking import re_ranking, naive_rerank  # disabled\n'
    're_ranking = naive_rerank = None'
)

with open(path, 'w') as f:
    f.write(src)
print("✓ Fixed. Re-run Cell 10.")

✓ Fixed. Re-run Cell 10.


In [26]:
path = '/content/SMS-Loss/scripts/test_mir.py'
with open(path) as f:
    src = f.read()

# Find and remove everything from train_dataset creation to the break,
# keeping only the val_stats call
old = (
    "    train_dataset = VideoCaptionDatasetCLIP(\n"
    "        args.dataset, args.root, args.train_metadata,\n"
    "        transform=train_transform, is_training=True, tokenizer=tokenizer,\n"
    "        clip_length=args.clip_length, clip_stride=args.clip_stride,\n"
    "        chunk_len=args.video_chunk_length,\n"
    "        threads=args.decode_threads,\n"
    "        fast_rrc=args.fused_decode_crop, rrc_params=(crop_size, (0.5, 1.0)),\n"
    "    )\n"
)
new = "    # train_dataset skipped — not needed for inference\n"
src = src.replace(old, new, 1)

# Also skip train_sampler, train_loader, lr_schedule, and the warm-up loop
import re
# Remove train_sampler
src = re.sub(r'    train_sampler = .*?\n', '    # train_sampler skipped\n', src, count=1)
# Remove train_loader + print
src = re.sub(
    r"    train_loader = torch\.utils\.data\.DataLoader\(\s+train_dataset.*?\)\n"
    r"    print\('len\(train_loader\).*?\n",
    "    # train_loader skipped\n",
    src, count=1, flags=re.DOTALL
)
# Remove lr_schedule
src = re.sub(r'    lr_schedule = cosine_scheduler\(.*?\)\n', '    # lr_schedule skipped\n', src, count=1, flags=re.DOTALL)
# Remove the warm-up loop (for data_iter ... break)
src = re.sub(
    r'    for data_iter, inputs in enumerate\(train_loader\):.*?break\n',
    '    # warm-up loop skipped\n',
    src, count=1, flags=re.DOTALL
)

with open(path, 'w') as f:
    f.write(src)
print("✓ Skipped train_dataset. Re-run Cell 10.")

✓ Skipped train_dataset. Re-run Cell 10.


In [27]:
import sys
sys.path.insert(0, '/content/SMS-Loss')
from avion.data.clip_dataset import VideoCaptionDatasetCLIP
import inspect
print(inspect.signature(VideoCaptionDatasetCLIP.__init__))

(self, args, metadata, transform=None, is_training=True, tokenizer=None, chunk_len=300, clip_length=32, clip_stride=2, threads=1, fast_rrc=False, rrc_params=(224, (0.5, 1.0)), fast_rcc=False, rcc_params=(224,), subsample_stride=None)


In [28]:
path = '/content/SMS-Loss/scripts/test_mir.py'
with open(path) as f:
    src = f.read()

old = (
    "    val_dataset = VideoCaptionDatasetCLIP(\n"
    "        args.dataset, args.root, args.val_metadata,\n"
    "        transform=val_transform, is_training=False, tokenizer=tokenizer,\n"
    "        clip_length=args.clip_length, clip_stride=args.clip_stride,\n"
    "        chunk_len=args.video_chunk_length,\n"
    "        fast_rcc=args.fused_decode_crop, rcc_params=(crop_size,),\n"
    "    )"
)
new = (
    "    val_dataset = VideoCaptionDatasetCLIP(\n"
    "        args, args.val_metadata,\n"
    "        transform=val_transform, is_training=False, tokenizer=tokenizer,\n"
    "        clip_length=args.clip_length, clip_stride=args.clip_stride,\n"
    "        chunk_len=args.video_chunk_length,\n"
    "        fast_rcc=args.fused_decode_crop, rcc_params=(crop_size,),\n"
    "    )"
)

assert old in src, "Pattern not found — print lines 290-305 of test_mir.py and share"
src = src.replace(old, new)
with open(path, 'w') as f:
    f.write(src)
print("✓ Fixed val_dataset call. Re-run Cell 10.")

✓ Fixed val_dataset call. Re-run Cell 10.


In [40]:
path = '/content/SMS-Loss/scripts/test_mir.py'
with open(path) as f:
    src = f.read()

old = (
    "                if args.flip == True:\n"
    "                    inputs_flip = inputs\n"
    "                    inputs_flip[0] = torch.flip(inputs[0], dims=[-1])\n"
    "                    image_features_flip, text_features_flip, logit_scale = model(*inputs_flip)\n"
    "                    image_features += image_features_flip\n"
    "                    text_features += text_features_flip\n"
)
new = (
    "                if args.flip:\n"
    "                    video_flip = torch.flip(inputs[0], dims=[-1])  # copy, not alias\n"
    "                    image_features_flip, _, _ = model(video_flip, inputs[1])\n"
    "                    image_features = F.normalize(image_features + image_features_flip, dim=-1)\n"
)

assert old in src, "Patch 1 pattern not found — check indentation or share lines 360-370"
src = src.replace(old, new)
with open(path, 'w') as f:
    f.write(src)
print("✓ Patch 1 applied: flip bug fixed.")

✓ Patch 1 applied: flip bug fixed.


In [41]:
path = '/content/SMS-Loss/scripts/test_mir.py'
with open(path) as f:
    src = f.read()

old = (
    "    #dual-softmax\n"
    "    # if True:\n"
    "    #     similarity_matrix = softmax_numpy(similarity_matrix / 500, dim=1) * similarity_matrix\n"
    "    #     similarity_matrix = softmax_numpy(similarity_matrix, dim=0)\n"
    "    # else:\n"
    "    similarity_matrix = (similarity_matrix_dist + 1) / 2\n"
)
new = (
    "    #dual-softmax\n"
    "    sim = similarity_matrix_dist / (similarity_matrix_dist.std() + 1e-6)\n"
    "    sim = softmax_numpy(sim, dim=1) * softmax_numpy(sim, dim=0)\n"
    "    sim_min, sim_max = sim.min(), sim.max()\n"
    "    similarity_matrix = (sim - sim_min) / (sim_max - sim_min + 1e-6)\n"
)

assert old in src, "Patch 2 pattern not found — print lines 404-412 of test_mir.py and share"
src = src.replace(old, new)
with open(path, 'w') as f:
    f.write(src)
print("✓ Patch 2 applied: dual softmax enabled.")

✓ Patch 2 applied: dual softmax enabled.


In [42]:
path = '/content/SMS-Loss/scripts/test_mir.py'
with open(path) as f:
    src = f.read()

old = (
    "    video_id = pd.read_csv(args.val_metadata).values[:, 0]\n"
    "    text_id = pd.read_csv(args.val_metadata.replace('test', 'test_sentence')).values[:, 0]\n"
    "    indexes = [video_id.tolist().index(elem) for elem in text_id]\n"
    "    similarity_matrix = similarity_matrix.T[:, indexes]\n"
)
new = (
    "    # alpha-QE: expand each query with its top-k retrieved neighbours\n"
    "    def alpha_qe(sim, alpha=3.0, top_k=10):\n"
    "        ranks = np.argsort(-sim, axis=1)\n"
    "        sim_qe = sim.copy()\n"
    "        for i in range(sim.shape[0]):\n"
    "            top_idx = ranks[i, :top_k]\n"
    "            weights = sim[i, top_idx] ** alpha          # (k,)\n"
    "            sim_qe[i] += alpha * (weights @ sim[top_idx]) / top_k\n"
    "        return sim_qe\n"
    "    similarity_matrix = alpha_qe(similarity_matrix, alpha=3.0, top_k=10)\n"
    "\n"
    "    video_id = pd.read_csv(args.val_metadata).values[:, 0]\n"
    "    text_id = pd.read_csv(args.val_metadata.replace('test', 'test_sentence')).values[:, 0]\n"
    "    indexes = [video_id.tolist().index(elem) for elem in text_id]\n"
    "    similarity_matrix = similarity_matrix.T[:, indexes]\n"
)

assert old in src, "Patch 3 pattern not found — print lines 410-416 of test_mir.py and share"
src = src.replace(old, new)
with open(path, 'w') as f:
    f.write(src)
print("✓ Patch 3 applied: alpha-QE added.")

✓ Patch 3 applied: alpha-QE added.


In [43]:
path = '/content/SMS-Loss/scripts/test_mir.py'
with open(path) as f:
    src = f.read()

# 4a: add the --num-crops argument to the parser
old_arg = (
    "    parser.add_argument('--flip', action='store_true', help='apply image(video) filp')\n"
)
new_arg = (
    "    parser.add_argument('--flip', action='store_true', help='apply image(video) filp')\n"
    "    parser.add_argument('--num-crops', default=1, type=int, help='number of spatial crops for TTA (1=center only, 3=left+center+right)')\n"
)

assert old_arg in src, "Patch 4a pattern not found — check --flip arg line"
src = src.replace(old_arg, new_arg)

# 4b: replace the single forward pass with multi-crop averaging
old_fwd = (
    "                # compute output\n"
    "                if args.fused_decode_crop and len(transform_gpu) > 0:\n"
    "                    inputs[0] = inputs[0].permute(0, 4, 1, 2, 3)\n"
    "                    inputs[0] = transform_gpu(inputs[0])\n"
    "                image_features, text_features, logit_scale = model(*inputs)\n"
)
new_fwd = (
    "                # compute output\n"
    "                if args.fused_decode_crop and len(transform_gpu) > 0:\n"
    "                    inputs[0] = inputs[0].permute(0, 4, 1, 2, 3)\n"
    "                    inputs[0] = transform_gpu(inputs[0])\n"
    "                # multi-crop TTA along the width dimension\n"
    "                if args.num_crops > 1:\n"
    "                    _v = inputs[0]  # (B, C, T, H, W)\n"
    "                    _w = _v.shape[-1]\n"
    "                    _cs = 224\n"
    "                    _step = max((_w - _cs) // (args.num_crops - 1), 0)\n"
    "                    _crops = [_v[..., i * _step: i * _step + _cs] for i in range(args.num_crops)]\n"
    "                    _feats = [model(c, inputs[1]) for c in _crops]\n"
    "                    image_features = F.normalize(\n"
    "                        sum(f[0] for f in _feats) / args.num_crops, dim=-1)\n"
    "                    text_features = _feats[0][1]\n"
    "                    logit_scale  = _feats[0][2]\n"
    "                else:\n"
    "                    image_features, text_features, logit_scale = model(*inputs)\n"
)

assert old_fwd in src, "Patch 4b pattern not found — print lines 356-362 of test_mir.py and share"
src = src.replace(old_fwd, new_fwd)

with open(path, 'w') as f:
    f.write(src)
print("✓ Patch 4 applied: multi-crop TTA added (use --num-crops 3 to activate).")

✓ Patch 4 applied: multi-crop TTA added (use --num-crops 3 to activate).


In [29]:
# ── Patch clip_dataset.py line 160 (relevancy_mat load) ──────────────────────
import re

path1 = "/content/SMS-Loss/avion/data/clip_dataset.py"
with open(path1, "r") as f:
    src = f.read()

old = "self.relevancy_mat = pickle.load(open(args.relevancy_test, 'rb'))"
new = (
    "import os as _os\n"
    "        _rel = getattr(args, 'relevancy_test', None) or getattr(args, 'relevancy_path', None)\n"
    "        self.relevancy_mat = pickle.load(open(_rel, 'rb')) if _rel and _os.path.exists(_rel) else None"
)

if old in src:
    src = src.replace(old, new)
    with open(path1, "w") as f:
        f.write(src)
    print("clip_dataset.py patched ✓")
else:
    print("clip_dataset.py: pattern not found — check manually")
    print("Looking for:", repr(old))

# ── Patch test_mir.py: add args.relevancy_test alias before val_dataset ───────
path2 = "/content/SMS-Loss/scripts/test_mir.py"
with open(path2, "r") as f:
    src2 = f.read()

# Anchor on the val_dataset creation line
anchor = "val_dataset = VideoCaptionDatasetCLIP("
alias  = "# alias missing arg\n    if not hasattr(args, 'relevancy_test'):\n        args.relevancy_test = None\n    "

if alias not in src2 and anchor in src2:
    src2 = src2.replace(anchor, alias + anchor)
    with open(path2, "w") as f:
        f.write(src2)
    print("test_mir.py patched ✓")
elif alias in src2:
    print("test_mir.py: already patched ✓")
else:
    print("test_mir.py: anchor not found — check manually")

clip_dataset.py patched ✓
test_mir.py patched ✓


In [30]:
# ── Re-patch clip_dataset.py (fix IndentationError from previous patch) ───────
path1 = "/content/SMS-Loss/avion/data/clip_dataset.py"
with open(path1, "r") as f:
    lines = f.readlines()

new_lines = []
i = 0
while i < len(lines):
    line = lines[i]
    stripped = line.rstrip()

    # Remove any broken patch lines we may have introduced
    if ("import os as _os" in stripped and "_rel" not in stripped and
            stripped.strip().startswith("import os as _os")):
        i += 1
        continue
    if "_rel = getattr(args, 'relevancy_test'" in stripped:
        i += 1
        continue
    if "self.relevancy_mat = pickle.load(open(_rel" in stripped:
        i += 1
        continue

    # Replace the original broken line (both old and any half-patched form)
    if ("self.relevancy_mat = pickle.load(open(args.relevancy_test" in stripped or
            "self.relevancy_mat = pickle.load(open(args.relevancy_path" in stripped):
        indent = len(line) - len(line.lstrip())
        pad = " " * indent
        new_lines.append(f"{pad}import os as _os\n")
        new_lines.append(f"{pad}_rel = getattr(args, 'relevancy_test', None) or getattr(args, 'relevancy_path', None)\n")
        new_lines.append(f"{pad}self.relevancy_mat = pickle.load(open(_rel, 'rb')) if _rel and _os.path.exists(_rel) else None\n")
        i += 1
        continue

    new_lines.append(line)
    i += 1

with open(path1, "w") as f:
    f.writelines(new_lines)

print("clip_dataset.py re-patched ✓")

# ── Verify no syntax errors ───────────────────────────────────────────────────
import ast
with open(path1) as f:
    src = f.read()
try:
    ast.parse(src)
    print("Syntax OK ✓")
except SyntaxError as e:
    print(f"Still broken at line {e.lineno}: {e.msg}")
    print("Context:", src.splitlines()[e.lineno-2:e.lineno+1])

clip_dataset.py re-patched ✓
Still broken at line 160: expected an indented block after 'elif' statement on line 159
Context: ["                elif 'test' in metadata:", '                else:', '                    raise ValueError(\'{} should contain either "train" or "test"!\'.format(metadata))']


In [31]:
with open("/content/SMS-Loss/avion/data/clip_dataset.py") as f:
    lines = f.readlines()
for i, l in enumerate(lines[150:175], start=151):
    print(f"{i:3d}| {l}", end="")

151|                     fps = fps_dict[osp.join(self.root, vid_path + '.MP4')]
152|                     # start_frame = int(np.round(fps * start_timestamp))
153|                     # end_frame = int(np.ceil(fps * end_timestamp))
154|                     self.samples.append((vid_path, start_timestamp, end_timestamp, fps, narration, verb, noun))
155|             if self.dataset == 'ek100_mir':
156|                 self.metadata_sentence = pd.read_csv(metadata[:metadata.index('.csv')] + '_sentence.csv')
157|                 if 'train' in metadata:
158|                     self.relevancy_mat = pickle.load(open(args.relevancy_train, 'rb'))
159|                 elif 'test' in metadata:
160|                 else:
161|                     raise ValueError('{} should contain either "train" or "test"!'.format(metadata))
162|                 self.relevancy = .1
163|         else:
164|             raise NotImplementedError
165|     
166|     def get_raw_item(
167|         self, i, is_training=Tr

In [32]:
path1 = "/content/SMS-Loss/avion/data/clip_dataset.py"
with open(path1, "r") as f:
    lines = f.readlines()

# Line 159 is index 158 (0-based), the empty elif block is lines 158-159
# We need to insert the body between elif and else
new_lines = []
i = 0
while i < len(lines):
    new_lines.append(lines[i])
    # After the empty `elif 'test' in metadata:` line, insert the body
    if lines[i].rstrip().endswith("elif 'test' in metadata:"):
        # Check next line is `else:` (empty elif body)
        if i + 1 < len(lines) and lines[i+1].rstrip().lstrip().startswith("else:"):
            indent = len(lines[i]) - len(lines[i].lstrip()) + 4  # one extra indent level
            pad = " " * indent
            new_lines.append(f"{pad}import os as _os\n")
            new_lines.append(f"{pad}_rel = getattr(args, 'relevancy_test', None) or getattr(args, 'relevancy_path', None)\n")
            new_lines.append(f"{pad}self.relevancy_mat = pickle.load(open(_rel, 'rb')) if _rel and _os.path.exists(_rel) else None\n")
    i += 1

with open(path1, "w") as f:
    f.writelines(new_lines)

# Verify
import ast
with open(path1) as f:
    src = f.read()
try:
    ast.parse(src)
    print("Syntax OK ✓")
except SyntaxError as e:
    print(f"SyntaxError at line {e.lineno}: {e.msg}")

# Show the fixed region
with open(path1) as f:
    lines = f.readlines()
for i, l in enumerate(lines[153:167], start=154):
    print(f"{i:3d}| {l}", end="")


Syntax OK ✓
154|                     self.samples.append((vid_path, start_timestamp, end_timestamp, fps, narration, verb, noun))
155|             if self.dataset == 'ek100_mir':
156|                 self.metadata_sentence = pd.read_csv(metadata[:metadata.index('.csv')] + '_sentence.csv')
157|                 if 'train' in metadata:
158|                     self.relevancy_mat = pickle.load(open(args.relevancy_train, 'rb'))
159|                 elif 'test' in metadata:
160|                     import os as _os
161|                     _rel = getattr(args, 'relevancy_test', None) or getattr(args, 'relevancy_path', None)
162|                     self.relevancy_mat = pickle.load(open(_rel, 'rb')) if _rel and _os.path.exists(_rel) else None
163|                 else:
164|                     raise ValueError('{} should contain either "train" or "test"!'.format(metadata))
165|                 self.relevancy = .1
166|         else:
167|             raise NotImplementedError


In [34]:
HAS_FLASH_ATTN = importlib.util.find_spec("flash_attn") is not None
flash_flag = "--use-flash-attn" if HAS_FLASH_ATTN else ""
print(f"flash_attn: {'enabled' if HAS_FLASH_ATTN else 'disabled'}")

flash_attn: enabled


In [ ]:
CKPT = f"{EXP_DIR}/checkpoint_0009.pt"   # full checkpoint, not _weights.pt

cmd = f"""\
torchrun --nproc_per_node=1 scripts/test_mir.py \\
  --root "{DATA_ROOT}" \\
  --train-metadata "{TRAIN_CSV}" \\
  --val-metadata   "{TEST_CSV}" \\
  --pretrain-model "{CKPT}" \\
  --model CLIP_VITL14 \\
  --project-embed-dim 256 \\
  --use-fast-conv1 \\
  {flash_flag} \\
  --flip \\
  --num-crops 3 \\
  --clip-length 32 \\
  --clip-stride 2 \\
  --batch-size 32 \\
  --output-dir "{EXP_DIR}"
"""
print(cmd)
!{cmd}

torchrun --nproc_per_node=1 scripts/test_mir.py \
  --root "/content/drive/MyDrive/EK100_320p_15sec_30fps_libx264" \
  --train-metadata "/content/drive/MyDrive/EK100_annotations/EPIC_100_retrieval_train.csv" \
  --val-metadata   "/content/drive/MyDrive/EK100_annotations/EPIC_100_retrieval_test.csv" \
  --pretrain-model "/content/drive/MyDrive/experiments/sms_vitl/checkpoint_0009.pt" \
  --model CLIP_VITL14 \
  --project-embed-dim 256 \
  --use-fast-conv1 \
  --use-flash-attn \
  --flip \
  --num-crops 3 \
  --clip-length 32 \
  --clip-stride 2 \
  --batch-size 32 \
  --output-dir "/content/drive/MyDrive/experiments/sms_vitl"

/content/SMS-Loss/scripts/test_mir.py:23: DeprecationWarning: `TorchScript` support for functional optimizers is deprecated and will be removed in a future PyTorch release. Consider using the `torch.compile` optimizer instead.
  from torch.distributed.optim import ZeroRedundancyOptimizer
/usr/local/lib/python3.12/dist-packages/torchvision/transforms/_functional_vi

In [38]:
!ls drive

ls: cannot access 'drive': No such file or directory


In [39]:
import zipfile, os, pickle, subprocess
import numpy as np

# ── Load your model's output ──────────────────────────────────────────────────
pkl_path = f"{EXP_DIR}/submission.pkl"
with open(pkl_path, "rb") as f:
    raw_out = pickle.load(f)

sim_mat = np.array(raw_out["sim_mat"], dtype=np.float32)
vis_ids = [str(v) for v in raw_out["vis_ids"]]
txt_ids = [str(t) for t in raw_out["txt_ids"]]

print(f"sim_mat : {sim_mat.shape}  dtype={sim_mat.dtype}")
print(f"vis_ids : {len(vis_ids)}  |  txt_ids : {len(txt_ids)}")

# ── SLS scores (adjust to match your model's supervision level) ───────────────
SLS_PT = 2   # pre-training supervision
SLS_TL = 3   # training labels
SLS_TD = 3   # training data

# ── Build a Python-3.7-compatible pickle ──────────────────────────────────────
def make_compat_pickle(sim_mat, vis_ids, txt_ids, sls_pt, sls_tl, sls_td):
    payload = {
        "version":   "0.1",
        "challenge": "multi_instance_retrieval",
        "sls_pt":    sls_pt,
        "sls_tl":    sls_tl,
        "sls_td":    sls_td,
        "sim_mat":   sim_mat,
        "vis_ids":   vis_ids,
        "txt_ids":   txt_ids,
    }
    raw = pickle.dumps(payload, protocol=2)
    # Fix numpy >= 2.0 vs grader's old numpy
    raw = raw.replace(b"numpy._core.multiarray", b"numpy.core.multiarray")
    return raw

# ── Write test.pkl (the grader requires exactly this name) ────────────────────
tmp_pkl = "/tmp/test.pkl"
pkl_bytes = make_compat_pickle(sim_mat, vis_ids, txt_ids, SLS_PT, SLS_TL, SLS_TD)
with open(tmp_pkl, "wb") as f:
    f.write(pkl_bytes)

# Sanity check
check = pickle.loads(pkl_bytes)
assert np.array(check["sim_mat"]).shape == sim_mat.shape, "Shape mismatch after re-load!"
print(f"test.pkl OK — {len(pkl_bytes)/1e6:.1f} MB")

# ── Zip it up ─────────────────────────────────────────────────────────────────
zip_path = f"{EXP_DIR}/submission.zip"
subprocess.run(
    f'cd /tmp && zip -j "{zip_path}" test.pkl',
    shell=True, check=True, capture_output=True
)

print(f"\nReady: {zip_path}  ({os.path.getsize(zip_path)/1e6:.1f} MB)")
print("Submit to: https://www.codabench.org/competitions/12008/")

sim_mat : (9668, 3842)  dtype=float32
vis_ids : 9668  |  txt_ids : 3842
test.pkl OK — 208.3 MB

Ready: /content/drive/MyDrive/experiments/sms_vitl/submission.zip  (157.0 MB)
Submit to: https://www.codabench.org/competitions/12008/
